In [0]:

# 1. Install the library
%pip install beautifulsoup4

# 2. Verify installation immediately
try:
    from bs4 import BeautifulSoup
    print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:
    print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")

In [0]:
# Setup & Dynamic Link Finder
# -------------------------------------------------------------------------
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, timedelta

# 1. Define URL
base_page_url = "https://www.dubaipulse.gov.ae/data/dld-transactions/dld_transactions-open"

# -------------------------------------------------------------------------
# 2. Azure Storage Paths (PASTE YOUR INFO BELOW)
# -------------------------------------------------------------------------
storage_account_name = "{storage account name}" 
storage_account_key = "{storage account key}"

# Connect to ADLS
spark.conf.set(
f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
storage_account_key
)

# Define Folders
landing_zone = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/transactions/landing/"
archive_zone = f"abfss://{container_name@{storage_account_name}.dfs.core.windows.net/transactions/archive/"

# 3. Dynamic Link Finder
def get_dynamic_download_url():
    print(f"🔍 Scraping {base_page_url}...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    response = requests.get(base_page_url, headers=headers)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
# Logic: Find link containing 'download' and 'transactions.csv'
    for link in soup.find_all('a', href=True):
        href = link['href']
        if "download" in href and "transactions.csv" in href:
            final_url = f"https://www.dubaipulse.gov.ae{href}" if href.startswith("/") else href
            print(f"✅ Found dynamic link: {final_url}")
            return final_url
    
    raise Exception("❌ Could not find the download link on the page.")

print("Configuration Loaded & Auto-Create Enabled.")

In [0]:
# Retention Policy
# -------------------------------------------------------------------------
def cleanup_old_files(days_to_keep=14):
    print(" Checking archive for old files...")
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)

    try:
        files = dbutils.fs.ls(archive_zone)
        for f in files:
            # Expected format: YYYY-MM-DD.csv
            filename = f.name.replace(".csv", "")
            try:
                file_date = datetime.strptime(filename, "%Y-%m-%d")
                if file_date < cutoff_date:
                    print(f" 🗑️ Deleting: {f.name}")
                    dbutils.fs.rm(f.path)
            except ValueError:
                continue
    except Exception:
        print(" Archive is empty or does not exist yet.")

cleanup_old_files()

In [0]:
# Setting up the  Archival Logic
# -------------------------------------------------------------------------
def archive_current_file():
    current_file_path = f"{landing_zone}current.csv"

    try:
# Check if file exists
        dbutils.fs.ls(current_file_path)

# Rename to YYYY-MM-DD.csv
        today_str = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone}{today_str}.csv"

        print(f"📦 Archiving previous 'current.csv' to: {archive_path}")
        dbutils.fs.mv(current_file_path, archive_path)

    except Exception:
        print("ℹ️ No 'current.csv' found in landing. First run?")

archive_current_file()

In [0]:
# CELL 5: Download Snapshot (Robust Version with Retries)
# -------------------------------------------------------------------------
import os
import time
import requests

def download_snapshot_with_retry(max_retries=3):
    fresh_url = get_dynamic_download_url()
    local_path = "/tmp/current.csv"
    headers = {'User-Agent': 'Mozilla/5.0'}

    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Attempt {attempt}/{max_retries}: Downloading to driver node...")

            # 30-second timeout, Stream=True
            with requests.get(fresh_url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(local_path, 'wb') as f:
                    # 10MB Chunks
                    for chunk in r.iter_content(chunk_size=10*1024*1024):
                        if chunk:
                            f.write(chunk)

            print("✅ Download complete.")
            break  # Exit loop on success

        except Exception as e:
            print(f"⚠️ Attempt {attempt} failed: {str(e)}")
            if attempt < max_retries:
                print("⏳ Waiting 5 seconds...")
                time.sleep(5)
            else:
                print("❌ All retries failed.")
                raise e

    # Upload to Data Lake
    if os.path.exists(local_path):
        print("⬆️ Uploading to Bronze Landing Zone...")
        dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
        os.remove(local_path)
        print("✅ Success! 'current.csv' is updated in the Data Lake.")

# Run it
download_snapshot_with_retry()

In [0]:
# CELL 2: The Master Ingestion Function
# -------------------------------------------------------------------------
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime

# -------------------------------------------------------------------------
# 1. Azure Storage Config (PASTE YOUR INFO BELOW)
# -------------------------------------------------------------------------
storage_account_name = "adlsdubaiYOURNAME"  # <-- PASTE Name here
storage_account_key = "YOUR_ACCESS_KEY_HERE"  # <-- PASTE Key here

# Connect to ADLS
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# Auto-create folders if missing
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

# -------------------------------------------------------------------------
# 2. The Reusable Ingestion Function
# -------------------------------------------------------------------------
def ingest_dataset(dataset_name, page_url, file_pattern):
    print(f"\n🚀 STARTING INGESTION: {dataset_name.upper()}")

    # Define Dynamic Paths (bronze/units, bronze/projects, etc.)
    base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/{dataset_name}"
    landing_zone = f"{base_path}/landing/"
    archive_zone = f"{base_path}/archive/"

    # A. Link Finder
    print(" 🔍 Finding download link on page...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(page_url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')

    fresh_url = None
    for link in soup.find_all('a', href=True):
        if "download" in link['href'] and file_pattern in link['href']:
            fresh_url = (
                f"https://www.dubaipulse.gov.ae{link['href']}"
                if link['href'].startswith("/")
                else link['href']
            )
            print(f" ✅ Found link: {fresh_url}")
            break

    if not fresh_url:
        raise Exception(f"❌ Link not found for {file_pattern}")

    # B. Archive Current File (if exists)
    try:
        dbutils.fs.ls(f"{landing_zone}current.csv")
        today = datetime.now().strftime("%Y-%m-%d")
        dbutils.fs.mv(
            f"{landing_zone}current.csv",
            f"{archive_zone}{today}.csv"
        )
        print(" 📦 Archived old file.")
    except Exception:
        pass

    # C. Download & Upload
    local_path = f"/tmp/{dataset_name}.csv"
    print(" ⬇️ Downloading & Uploading...")

    with requests.get(fresh_url, headers=headers, stream=True) as r:
        r.raise_for_status()
        with open(local_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=10 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

    dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
    os.remove(local_path)

    print(f"✅ FINISHED: {dataset_name} is ready.")

print("Master Function Loaded.")

In [0]:
# CELL 3: Execute Ingestion
# -------------------------------------------------------------------------

# 1. Ingest Units (Product Dimension)
ingest_dataset(
dataset_name="units",
page_url="https://www.dubaipulse.gov.ae/data/dld-registration/dld_units-open",
file_pattern="units.csv"
)

# 2. Ingest Projects (Status Dimension)
ingest_dataset(
dataset_name="projects",
page_url="https://www.dubaipulse.gov.ae/data/dld-registration/dld_projects-open",
file_pattern="projects.csv"
)

In [0]:
# Commit and push